# 03 – Model 2: Complex CNN tự thiết kế (dùng CNN block)
**Người phụ trách:** Thành viên B &nbsp;|&nbsp; **Requirement:** Req 3 – Design your own complex CNN using CNN blocks, convolutional, pooling, and fully connected layers

**Mục tiêu:** Thiết kế CNN gồm nhiều **CNN block** lặp lại (ý tưởng block giống VGG trong phần Modern CNN), thử 3 biến thể và chọn bản tốt nhất theo validation.

**Cần có trước:** `data/splits.csv` (notebook 01).

**File tạo ra:** `models/complex_cnn_best.keras` (+ 3 biến thể), `results/complex_cnn_*_history.csv`, `results/complex_cnn_*_summary.json`, `results/complex_cnn_variants.csv`, `figures/complex_cnn_curve.png`

> Chạy lần lượt từng cell từ trên xuống (`Shift + Enter`). Ô có chữ **✍️ NHẬN XÉT** là chỗ bạn tự viết phân tích.

## 0. Thiết lập môi trường
Cell này làm 3 việc:
1. **Trên Colab:** kết nối Google Drive (để lưu model và kết quả lâu dài) và tải bộ dữ liệu EuroSAT (~95 MB) về máy Colab.
2. **Trên máy cá nhân:** dùng luôn thư mục dự án (notebook nằm trong `notebooks/`, ảnh nằm trong `data/EuroSAT_RGB/`).
3. Tạo sẵn các thư mục lưu kết quả: `data/`, `models/`, `results/`, `figures/`.

> Trên Colab nhớ bật GPU trước: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# ==============================================================================
# 0. THIẾT LẬP MÔI TRƯỜNG – luôn chạy cell này đầu tiên
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules                 # True nếu đang chạy trên Google Colab

if IN_COLAB:
    from google.colab import drive                        # thư viện kết nối Google Drive của Colab
    drive.mount("/content/drive")                         # hiện hộp thoại xin quyền truy cập Drive
    PROJECT_DIR = "/content/drive/MyDrive/eurosat_project"   # thư mục Drive DÙNG CHUNG của cả nhóm
    DATA_DIR = "/content/EuroSAT_RGB"                     # ảnh gốc để trên máy Colab (không lưu vào Drive)
    if not os.path.exists(DATA_DIR):                      # chỉ tải 1 lần cho mỗi phiên Colab
        os.system('wget -q -O /content/EuroSAT_RGB.zip "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1"')
        os.system("unzip -q -o /content/EuroSAT_RGB.zip -d /content/")   # giải nén -> /content/EuroSAT_RGB/
else:
    PROJECT_DIR = os.path.abspath("..")                   # thư mục gốc của dự án (cha của notebooks/)
    DATA_DIR = os.path.join(PROJECT_DIR, "data", "EuroSAT_RGB")

# Đường dẫn các file/thư mục kết quả – giống hệt cấu trúc repo GitHub
SPLITS_CSV  = os.path.join(PROJECT_DIR, "data", "splits.csv")   # cách chia train/val/test dùng chung
MODELS_DIR  = os.path.join(PROJECT_DIR, "models")               # model đã train (.keras)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")              # bảng số liệu (.csv, .json, .txt)
FIGURES_DIR = os.path.join(PROJECT_DIR, "figures")              # hình vẽ (.png)
for folder in [os.path.dirname(SPLITS_CSV), MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(folder, exist_ok=True)                    # tạo thư mục nếu chưa có

assert os.path.exists(DATA_DIR), f"Không tìm thấy ảnh tại {DATA_DIR} – kiểm tra lại bước tải/giải nén dữ liệu"
print("Ảnh gốc       :", DATA_DIR)
print("Số thư mục lớp:", len([d for d in os.listdir(DATA_DIR) if not d.startswith(".")]))
print("Lưu kết quả tại:", PROJECT_DIR)

In [ ]:
# ==============================================================================
# 1. IMPORT THƯ VIỆN VÀ HẰNG SỐ DÙNG CHUNG
# ==============================================================================
import json                                   # đọc/ghi file .json (tóm tắt kết quả train)
import time                                   # đo thời gian train
import numpy as np                            # xử lý mảng số
import pandas as pd                           # xử lý bảng dữ liệu
import matplotlib.pyplot as plt               # vẽ hình
import seaborn as sns                         # vẽ heatmap đẹp hơn
from PIL import Image                         # đọc file ảnh .jpg

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     BatchNormalization, Activation, GlobalAveragePooling2D,
                                     RandomFlip, RandomBrightness, Resizing, Rescaling)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42                             # seed cố định để kết quả lặp lại được
tf.keras.utils.set_random_seed(RANDOM_STATE)  # cố định seed cho numpy, python và tensorflow

CLASS_NAMES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
               "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]   # 10 lớp, theo thứ tự label_id 0..9
NUM_CLASSES = len(CLASS_NAMES)                # = 10
IMG_SIZE = 64                                 # ảnh EuroSAT có kích thước 64 x 64
BATCH_SIZE = 64                               # số ảnh mỗi lần cập nhật trọng số
EPOCHS = 30                                   # số epoch tối đa
PATIENCE = 5                                  # early stopping: chờ 5 epoch không cải thiện thì dừng

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "KHÔNG CÓ GPU – vào Runtime → Change runtime type → T4 GPU")

## 2. Nạp dữ liệu theo `splits.csv`
- Đọc file `data/splits.csv` (do notebook 01 tạo) để biết ảnh nào thuộc **train / val / test** – cả nhóm dùng chung một cách chia.
- Đọc từng ảnh thành mảng numpy, gộp lại thành tensor 4 chiều **(N, 64, 64, 3)** giống cách làm với MNIST trong bài giảng.
- **Chuẩn hóa:** chia 255 để đưa pixel từ [0, 255] về [0, 1].
- **Nhãn:** chuyển sang **one-hot** (vd. lớp 3 → `[0,0,0,1,0,0,0,0,0,0]`) để dùng với `categorical_crossentropy`.

In [ ]:
# ==============================================================================
# 2. NẠP DỮ LIỆU THEO splits.csv
# ==============================================================================
splits = pd.read_csv(SPLITS_CSV)              # bảng gồm: filepath, label, label_id, split

def load_split(split_name):
    """Đọc toàn bộ ảnh của 1 tập (train/val/test) -> (X, y)."""
    df = splits[splits["split"] == split_name]                                   # lọc các dòng của tập này
    X = np.stack([np.array(Image.open(os.path.join(DATA_DIR, fp))) for fp in df["filepath"]])   # (N, 64, 64, 3)
    y = df["label_id"].values                                                    # nhãn dạng số 0..9
    return X, y

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")

X_train = X_train.astype("float32") / 255.0      # chuẩn hóa pixel về [0, 1]
X_val   = X_val.astype("float32") / 255.0

Y_train = to_categorical(y_train, NUM_CLASSES)   # nhãn one-hot, vd. 3 -> [0,0,0,1,0,0,0,0,0,0]
Y_val   = to_categorical(y_val, NUM_CLASSES)

print("X_train:", X_train.shape, "| Y_train:", Y_train.shape)
print("X_val  :", X_val.shape,   "| Y_val  :", Y_val.shape)

## 3. Data augmentation (chỉ hoạt động khi train)
Dùng các **lớp tiền xử lý của Keras** đặt ở đầu model, nên augmentation **tự động tắt** khi đánh giá/dự đoán.
- `RandomFlip("horizontal_and_vertical")`: lật ngang / dọc ngẫu nhiên. Ảnh vệ tinh chụp từ trên xuống, không có hướng "lên trên" nên lật không làm đổi ý nghĩa.
- `RandomBrightness(0.1)`: tăng/giảm độ sáng tối đa 10% – mô phỏng khác biệt ánh sáng, khí quyển.
- **Không dùng:** zoom mạnh (mỗi pixel = 10 m cố định, zoom làm sai kích thước ruộng, nhà, đường), xoay góc lẻ (tạo góc đen), đổi màu (màu nước, cây mang ý nghĩa).

> Augmentation giống hệt nhau ở cả 3 model để so sánh công bằng.

In [ ]:
# ==============================================================================
# 3. DATA AUGMENTATION – giống hệt nhau ở cả 3 model
# ==============================================================================
data_augmentation = Sequential([
    RandomFlip("horizontal_and_vertical"),             # lật ngang và/hoặc lật dọc ngẫu nhiên
    RandomBrightness(factor=0.1, value_range=(0, 1)),  # đổi độ sáng ±10% (ảnh đã ở thang 0–1)
], name="data_augmentation")

## 4. Thiết kế CNN block
Một **CNN block** của nhóm gồm:

`[Conv2D 3×3 → BatchNormalization → ReLU] × 2 → MaxPooling 2×2 → Dropout`

- **2 lớp Conv 3×3 liên tiếp:** nhìn được vùng 5×5 nhưng ít tham số hơn 1 lớp Conv 5×5 (ý tưởng của VGG).
- **BatchNormalization:** chuẩn hóa đầu ra mỗi lớp → train nhanh và ổn định hơn.
- **MaxPooling:** giảm kích thước 1 nửa sau mỗi block.
- **Dropout:** tắt ngẫu nhiên một phần nơ-ron để chống overfit.

**Kiến trúc tổng:** 4 block với số bộ lọc tăng dần 32 → 64 → 128 → 256, kích thước giảm dần 64 → 32 → 16 → 8 → 4.
Phần cuối dùng **GlobalAveragePooling** (lấy trung bình mỗi kênh) thay cho Flatten để giảm mạnh số tham số, rồi tới các lớp **fully connected**.

In [ ]:
# ==============================================================================
# 4. HÀM TẠO CNN BLOCK VÀ HÀM TẠO TOÀN BỘ MODEL
# ==============================================================================
def add_cnn_block(model, filters, use_batchnorm=True, dropout=0.25):
    """Thêm 1 CNN block vào model: [Conv 3x3 -> (BN) -> ReLU] x 2 -> MaxPool -> Dropout."""
    for _ in range(2):                                            # 2 lớp convolution liên tiếp
        model.add(Conv2D(filters, (3, 3), padding="same"))        # padding="same": giữ nguyên chiều cao, rộng
        if use_batchnorm:
            model.add(BatchNormalization())                       # chuẩn hóa đầu ra của lớp conv
        model.add(Activation("relu"))                             # hàm kích hoạt ReLU
    model.add(MaxPooling2D(pool_size=(2, 2)))                     # giảm kích thước 1 nửa
    model.add(Dropout(dropout))                                   # chống overfit


def build_complex_cnn(name="complex_cnn", use_batchnorm=True, use_gap=True):
    """use_batchnorm: có dùng BatchNormalization không.
    use_gap: True = GlobalAveragePooling, False = Flatten (để so sánh)."""
    model = Sequential(name=name)
    model.add(Input(shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(data_augmentation)

    for filters in [32, 64, 128, 256]:                            # 4 block, số bộ lọc tăng dần
        add_cnn_block(model, filters, use_batchnorm)              # 64 -> 32 -> 16 -> 8 -> 4

    if use_gap:
        model.add(GlobalAveragePooling2D())                       # 4x4x256 -> vector 256 phần tử
    else:
        model.add(Flatten())                                      # 4x4x256 -> vector 4.096 phần tử
    model.add(Dense(256, activation="relu"))                      # fully connected
    model.add(Dropout(0.5))
    model.add(Dense(NUM_CLASSES, activation="softmax"))           # đầu ra 10 lớp
    return model


build_complex_cnn().summary()

**✍️ NHẬN XÉT:** _Tổng số tham số? Kích thước ảnh sau mỗi block? So với Simple CNN, model này sâu hơn bao nhiêu lớp, nhiều hay ít tham số hơn?_

## 5. Train 3 biến thể để chứng minh các lựa chọn thiết kế
| Biến thể | Khác gì bản chính | Muốn trả lời câu hỏi |
|---|---|---|
| `complex_cnn_v1` | Bản chính (BN + GAP) | — |
| `complex_cnn_no_bn` | Bỏ BatchNormalization | BatchNorm có giúp không? |
| `complex_cnn_flatten` | Flatten thay GAP | GAP có tốt hơn Flatten không? |

Mỗi biến thể khoảng 5–15 phút trên GPU T4. Nếu Colab ngắt giữa chừng: chạy lại cell 0 → 4, thêm `#` trước biến thể đã xong rồi chạy lại cell này.

In [ ]:
# ==============================================================================
# HÀM TRAIN DÙNG CHUNG: train model + lưu model, lịch sử train và tóm tắt
# ==============================================================================
def train_and_save(model, name, learning_rate, epochs, initial_epoch=0):
    """Compile, train và lưu kết quả của 1 model.
    name: tên dùng để đặt tên file, vd 'simple_cnn'."""
    model.compile(optimizer=Adam(learning_rate=learning_rate),      # thuật toán tối ưu Adam
                  loss="categorical_crossentropy",                  # hàm mất mát cho nhãn one-hot
                  metrics=["accuracy"])                             # theo dõi accuracy khi train

    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE,   # dừng nếu val_loss không giảm sau PATIENCE epoch
                               restore_best_weights=True)              # và quay về trọng số tốt nhất
    checkpoint = ModelCheckpoint(os.path.join(MODELS_DIR, f"{name}.keras"),  # lưu model tốt nhất vào Drive
                                 monitor="val_loss", save_best_only=True)

    start = time.time()                                             # bắt đầu bấm giờ
    H = model.fit(X_train, Y_train,                                 # dữ liệu train
                  validation_data=(X_val, Y_val),                   # dữ liệu validation để theo dõi
                  batch_size=BATCH_SIZE, epochs=epochs, initial_epoch=initial_epoch,
                  callbacks=[early_stop, checkpoint], verbose=1)
    train_time = time.time() - start                                # tổng thời gian train (giây)

    history = pd.DataFrame(H.history)                               # loss, accuracy, val_loss, val_accuracy theo epoch
    history.insert(0, "epoch", [e + 1 for e in H.epoch])            # thêm cột số thứ tự epoch
    history.to_csv(os.path.join(RESULTS_DIR, f"{name}_history.csv"), index=False)

    best = history.loc[history["val_loss"].idxmin()]                # epoch có val_loss nhỏ nhất
    summary = {"model": name,
               "params": int(model.count_params()),                 # tổng số tham số
               "epochs_run": int(len(history)),
               "best_epoch": int(best["epoch"]),
               "best_val_loss": round(float(best["val_loss"]), 4),
               "best_val_accuracy": round(float(best["val_accuracy"]), 4),
               "train_time_sec": round(train_time, 1),
               "learning_rate": learning_rate}
    with open(os.path.join(RESULTS_DIR, f"{name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)                             # lưu tóm tắt ra file .json
    print(json.dumps(summary, indent=2))
    return history

In [ ]:
# ==============================================================================
# HÀM VẼ LEARNING CURVE (loss và accuracy theo epoch)
# ==============================================================================
def plot_history(history, title, save_name, mark_epoch=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))                 # 2 hình cạnh nhau
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(history["epoch"], history[metric], label=f"train {metric}")
        ax.plot(history["epoch"], history[f"val_{metric}"], label=f"val {metric}")
        if mark_epoch:                                              # vạch đánh dấu (vd. lúc bắt đầu fine-tune)
            ax.axvline(mark_epoch, color="gray", linestyle="--", label="bắt đầu fine-tune")
        ax.set_xlabel("epoch"); ax.set_title(metric); ax.legend()
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, save_name), dpi=150)      # lưu hình vào figures/
    plt.show()

In [ ]:
# ==============================================================================
# 5. TRAIN 3 BIẾN THỂ
# ==============================================================================
VARIANTS = {
    "complex_cnn_v1":      dict(use_batchnorm=True,  use_gap=True),    # bản chính
    "complex_cnn_no_bn":   dict(use_batchnorm=False, use_gap=True),    # bỏ BatchNorm
    "complex_cnn_flatten": dict(use_batchnorm=True,  use_gap=False),   # Flatten thay GAP
}
for name, config in VARIANTS.items():
    print(f"\n========== {name} ==========")
    tf.keras.utils.set_random_seed(RANDOM_STATE)                  # cùng seed cho mọi biến thể
    model = build_complex_cnn(name=name, **config)
    train_and_save(model, name, learning_rate=1e-3, epochs=EPOCHS)

## 6. So sánh 3 biến thể và chọn bản tốt nhất (theo `val_loss`, KHÔNG dùng test)

In [ ]:
# ==============================================================================
# 6. SO SÁNH VÀ CHỌN BẢN TỐT NHẤT
# ==============================================================================
import shutil

rows = []
for name, config in VARIANTS.items():
    with open(os.path.join(RESULTS_DIR, f"{name}_summary.json")) as f:
        rows.append({**json.load(f), **config})                     # gộp kết quả + cấu hình
variants_df = pd.DataFrame(rows)
variants_df.to_csv(os.path.join(RESULTS_DIR, "complex_cnn_variants.csv"), index=False)

best_name = variants_df.loc[variants_df["best_val_loss"].idxmin(), "model"]   # val_loss nhỏ nhất
shutil.copy(os.path.join(MODELS_DIR, f"{best_name}.keras"),
            os.path.join(MODELS_DIR, "complex_cnn_best.keras"))            # bản dùng ở notebook 05
shutil.copy(os.path.join(RESULTS_DIR, f"{best_name}_history.csv"),
            os.path.join(RESULTS_DIR, "complex_cnn_best_history.csv"))
print("Bản tốt nhất:", best_name)
variants_df[["model", "use_batchnorm", "use_gap", "params", "best_epoch",
             "best_val_loss", "best_val_accuracy", "train_time_sec"]]

In [ ]:
best_history = pd.read_csv(os.path.join(RESULTS_DIR, "complex_cnn_best_history.csv"))
plot_history(best_history, f"Complex CNN – {best_name}", "complex_cnn_curve.png")

**✍️ NHẬN XÉT:** _Bỏ BatchNorm thì val accuracy thay đổi thế nào? Flatten so với GAP: số tham số và val accuracy khác ra sao? Vì sao Complex CNN tốt hơn Simple CNN? (Tham khảo: khoảng 95–98%)._

## ✅ Xong notebook 03
Nhớ vẽ sơ đồ kiến trúc bằng draw.io → lưu `figures/complex_cnn_architecture.png`.